In [1]:
!pip install --upgrade numpy scikit-learn

  Using cached numpy-2.2.6-cp310-cp310-macosx_14_0_arm64.whl.metadata (62 kB)
  Using cached scikit_learn-1.7.2-cp310-cp310-macosx_12_0_arm64.whl.metadata (11 kB)
Using cached numpy-2.2.6-cp310-cp310-macosx_14_0_arm64.whl (5.3 MB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.7/8.7 MB 3.4 MB/s  0:00:02 eta 0:00:01
  Attempting uninstall: numpy
    Found existing installation: numpy 2.0.2
    Uninstalling numpy-2.0.2:
      Successfully uninstalled numpy-2.0.2
  Attempting uninstall: scikit-learn━━━━━━━━━━━━ 0/2 [numpy]
    Found existing installation: scikit-learn 1.0.232m0/2 [numpy]
    Uninstalling scikit-learn-1.0.2:━━━━━━━━ 0/2 [numpy]
      Successfully uninstalled scikit-learn-1.0.20/2 [numpy]
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [scikit-learn] [scikit-learn]
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
matplotlib 3.8.3 requires numpy<2,>=1.

In [2]:
import pandas as pd

In [3]:
df = pd.read_csv('deepface_predictions-5.csv')

# Split filename into parts
parts = df["filename"].str.split("_", expand=True)

# Extract numeric codes
df["face_id"] = parts[1]
df["race_code"] = parts[2].astype(int)
df["age_code"] = parts[3].astype(int)
df["gender_code"] = parts[4].astype(int)

In [4]:
df.head(5)

,filename,age,gender,race,face_id,race_code,age_code,gender_code
0,fairface_4394_0_7_0_val.jpg,32,Man,asian,4394,0,7,0
1,fairface_707_5_3_0_val.jpg,26,Man,middle eastern,707,5,3,0
2,fairface_2303_2_5_1_val.jpg,33,Woman,black,2303,2,5,1
3,fairface_8241_1_6_1_val.jpg,39,Man,indian,8241,1,6,1
4,fairface_2599_0_3_0_val.jpg,30,Man,asian,2599,0,3,0


In [5]:
import pandas as pd

# Example dataframe
# df has: filename, age, gender, race, age_code, gender_code, race_code

# Mappings
race_map = {
    0: "asian", 1: "indian", 2: "black", 3: "white",
    4: "middle eastern", 5: "latino hispanic"
}
age_map = {
    0: "0-2", 1: "3-9", 2: "10-19", 3: "20-29",
    4: "30-39", 5: "40-49", 6: "50-59", 7: "60-69", 8: "+70"
}
gender_map = {0: "Man", 1: "Woman"}

# Inverse mappings
race_inv = {v.lower(): k for k, v in race_map.items()}
gender_inv = {v.lower(): k for k, v in gender_map.items()}

# Function: convert raw age int → encoded int
def encode_age(age: int) -> int:
    if 0 <= age <= 2: return 0
    elif 3 <= age <= 9: return 1
    elif 10 <= age <= 19: return 2
    elif 20 <= age <= 29: return 3
    elif 30 <= age <= 39: return 4
    elif 40 <= age <= 49: return 5
    elif 50 <= age <= 59: return 6
    elif 60 <= age <= 69: return 7
    else: return 8   # 70+

# Encode predictions
df["pred_gender_code"] = df["gender"].str.lower().map(gender_inv)
df["pred_race_code"]   = df["race"].str.lower().map(race_inv)
df["pred_age_code"]    = df["age"].apply(encode_age)

df.head()

,filename,age,gender,race,face_id,race_code,age_code,gender_code,pred_gender_code,pred_race_code,pred_age_code
0,fairface_4394_0_7_0_val.jpg,32,Man,asian,4394,0,7,0,0,0,4
1,fairface_707_5_3_0_val.jpg,26,Man,middle eastern,707,5,3,0,0,4,3
2,fairface_2303_2_5_1_val.jpg,33,Woman,black,2303,2,5,1,1,2,4
3,fairface_8241_1_6_1_val.jpg,39,Man,indian,8241,1,6,1,0,1,4
4,fairface_2599_0_3_0_val.jpg,30,Man,asian,2599,0,3,0,0,0,4


In [6]:
df.loc[df['race_code'] == 6, 'race_code'] = 0

In [7]:
df.head()

,filename,age,gender,race,face_id,race_code,age_code,gender_code,pred_gender_code,pred_race_code,pred_age_code
0,fairface_4394_0_7_0_val.jpg,32,Man,asian,4394,0,7,0,0,0,4
1,fairface_707_5_3_0_val.jpg,26,Man,middle eastern,707,5,3,0,0,4,3
2,fairface_2303_2_5_1_val.jpg,33,Woman,black,2303,2,5,1,1,2,4
3,fairface_8241_1_6_1_val.jpg,39,Man,indian,8241,1,6,1,0,1,4
4,fairface_2599_0_3_0_val.jpg,30,Man,asian,2599,0,3,0,0,0,4


In [8]:
from sklearn.metrics import accuracy_score, classification_report

# --- Accuracy scores ---
gender_acc = accuracy_score(df["gender_code"], df["pred_gender_code"])
race_acc   = accuracy_score(df["race_code"], df["pred_race_code"])
age_acc    = accuracy_score(df["age_code"], df["pred_age_code"])

print(f"Gender Accuracy: {gender_acc:.3f}")
print(f"Race Accuracy:   {race_acc:.3f}")
print(f"Age Accuracy:    {age_acc:.3f}")

# --- Detailed reports ---
print("\n--- Gender Report ---")
print(classification_report(
    df["gender_code"], df["pred_gender_code"],
    target_names=list(gender_map.values())
))

print("\n--- Race Report ---")
print(classification_report(
    df["race_code"], df["pred_race_code"],
    target_names=list(race_map.values())
))

print("\n--- Age Report ---")
print(classification_report(
    df["age_code"], df["pred_age_code"],
    target_names=list(age_map.values())
))


Gender Accuracy: 0.690
Race Accuracy:   0.657
Age Accuracy:    0.289

--- Gender Report ---
              precision    recall  f1-score   support

         Man       0.64      0.97      0.77      1077
       Woman       0.91      0.38      0.53       958

    accuracy                           0.69      2035
   macro avg       0.78      0.67      0.65      2035
weighted avg       0.77      0.69      0.66      2035


--- Race Report ---
                 precision    recall  f1-score   support

          asian       0.75      0.87      0.81       535
         indian       0.81      0.42      0.55       293
          black       0.72      0.87      0.78       284
          white       0.57      0.78      0.66       389
 middle eastern       0.58      0.45      0.50       213
latino hispanic       0.46      0.32      0.38       321

       accuracy                           0.66      2035
      macro avg       0.65      0.62      0.61      2035
   weighted avg       0.66      0.66      0.6

/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this beha

In [9]:
# Gender accuracy
gender_acc = (df["gender_code"] == df["pred_gender_code"]).mean()

# Race accuracy
race_acc = (df["race_code"] == df["pred_race_code"]).mean()

# Age accuracy
age_acc = (df["age_code"] == df["pred_age_code"]).mean()

print(f"Gender Accuracy: {gender_acc:.3f}")
print(f"Race Accuracy:   {race_acc:.3f}")
print(f"Age Accuracy:    {age_acc:.3f}")


Gender Accuracy: 0.690
Race Accuracy:   0.657
Age Accuracy:    0.289
